# Native-space V1 encoding model with StudyForrest + DataLad

This notebook sets up a **within-subject encoding analysis** for the StudyForrest 3 T audio-visual movie. It asks whether emotion or body-contact annotations add held-out predictive power for voxels in V1 after accounting for low-level movie properties.

The workflow:

1. clones OpenNeuro `ds000113` directly from its official Git/DataLad repository;
2. checks out OpenNeuro snapshot tag `1.3.0` and pins auxiliary repositories to known commits;
3. retrieves only one subject's eight movie runs and required metadata;
4. motion-aligns runs and constructs a bilateral V1 mask in a subject-native BOLD grid;
5. aligns low-level, emotion, and body-contact annotations to each TR;
6. compares nested ridge-encoding models with leave-one-run-out cross-validation.

> This is a scientifically useful starting pipeline, not a finished confirmatory analysis. In particular, a strong visual encoding baseline should eventually include modern visual features (for example, motion-energy or CNN features). The supplied brightness/change annotations are a minimum control, not a complete account of visual input.

## Why these data and this space?

- Use the **3 T audio-visual movie** (`task-avmovie`). Emotion and body contact are properties of the visible film; the 7 T Forrest Gump acquisition is audio-only and is therefore not suitable for a visual V1 question.
- Retrieve `sub-XX/ses-movie/func/*_bold.nii.gz` from OpenNeuro, then motion-align all eight runs to run 1. This creates one **subject-native 3 T BOLD grid** without MNI or group normalization.
- Define V1 only from subject 1's measured polar-angle and eccentricity maps. The released maps are in the subject-specific `bold3Tp2` retinotopy grid, so the V1 boundary is drawn there and registered once into the movie-BOLD grid.
- Treat subjects separately. Native-space voxel identities do not correspond across participants.

## 1. Requirements

System tools:

- Git, git-annex, and DataLad
- FSL (`fslroi`, `fslmaths`, `mcflirt`, `flirt`, and preferably FSLeyes) for native-space alignment, map inspection, and V1 masking

Python packages are installed separately from the data. Restart the kernel if the install cell requests it.

In [ ]:
# Uncomment if needed:
# %pip install -q datalad nibabel nilearn pandas numpy scipy scikit-learn matplotlib seaborn


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nilearn.maskers import NiftiMasker
from nilearn.plotting import plot_roi, plot_stat_map
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

required_commands = ["git", "git-annex", "datalad", "fslroi", "fslmaths", "mcflirt", "flirt"]
optional_commands = ["fsleyes"]
missing_commands = []
for command in required_commands + optional_commands:
    location = shutil.which(command)
    print(f"{command:18s}", location or "NOT FOUND")
    if command in required_commands and location is None:
        missing_commands.append(command)
if missing_commands:
    raise EnvironmentError(
        "Install/configure the required system tools before continuing: "
        + ", ".join(missing_commands)
    )

## 2. Configuration and immutable dataset pins

OpenNeuro snapshots are Git tags. Auxiliary StudyForrest repositories are checked out at explicit commits rather than moving default branches.

In [ ]:
OPENNEURO_URL = "https://github.com/OpenNeuroDatasets/ds000113.git"
OPENNEURO_TAG = "1.3.0"
OPENNEURO_COMMIT = "35252aa8e7eab533e7a346d7eb6809aa6081a61d"

ANNOTATIONS_URL = "https://github.com/psychoinformatics-de/studyforrest-data-annotations.git"
ANNOTATIONS_COMMIT = "155097f944f9dc5a8790dd9148016508ab8b18ad"
BODYCONTACT_URL = "https://github.com/psychoinformatics-de/studyforrest-paper-bodycontactannotation.git"
BODYCONTACT_COMMIT = "ad42e8aab4a03d0f377ad4db7a9318094f7fc5d4"
RETINOTOPY_URL = "https://github.com/psychoinformatics-de/studyforrest-data-retinotopy.git"
RETINOTOPY_COMMIT = "6629c2f2f48e63899e36126087b8deed4ce90153"
TRANSFORMS_URL = "https://github.com/psychoinformatics-de/studyforrest-data-templatetransforms.git"
TRANSFORMS_COMMIT = "0afa47d792c28d493a79ca8854c4ffa1a015347f"

# git-annex creates local bookkeeping commits even for read-only data access.
# These variables affect only subprocesses launched by this notebook kernel.
GIT_IDENTITY_NAME = "StudyForrest notebook"
GIT_IDENTITY_EMAIL = "junruz@andrew.cmu.edu"
os.environ["GIT_AUTHOR_NAME"] = GIT_IDENTITY_NAME
os.environ["GIT_AUTHOR_EMAIL"] = GIT_IDENTITY_EMAIL
os.environ["GIT_COMMITTER_NAME"] = GIT_IDENTITY_NAME
os.environ["GIT_COMMITTER_EMAIL"] = GIT_IDENTITY_EMAIL
os.environ.setdefault("FSLOUTPUTTYPE", "NIFTI_GZ")

DATA_ROOT = Path("ds000113").resolve()
AUX_ROOT = Path("studyforrest-aux").resolve()
ANNOTATIONS_ROOT = AUX_ROOT / "annotations"
BODYCONTACT_ROOT = AUX_ROOT / "bodycontact"
RETINOTOPY_ROOT = AUX_ROOT / "retinotopy"
TRANSFORMS_ROOT = AUX_ROOT / "template-transforms"
ANALYSIS_ROOT = Path("studyforrest-v1-encoding").resolve()
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

PHASE2_SUBJECTS = ("sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-09", "sub-10", "sub-14", "sub-15", "sub-16", "sub-17", "sub-18", "sub-19", "sub-20")
SUBJECT = "sub-01"       # phase-2 subjects are not a consecutive 01..20 set
RUNS = tuple(range(1, 9))
TR = 2.0
RIDGE_ALPHA = 100.0
HRF_LAGS_TR = (2, 3, 4)     # 4, 6, and 8 seconds

assert SUBJECT in PHASE2_SUBJECTS, f"Choose a phase-2 subject: {PHASE2_SUBJECTS}"
print("OpenNeuro data:", DATA_ROOT)
print("Auxiliary datasets:", AUX_ROOT)
print("Analysis outputs:", ANALYSIS_ROOT)

In [ ]:
def run_command(args, cwd=None, check=True):
    """Run a command without invoking a shell; print it for provenance."""
    args = [str(arg) for arg in args]
    print("$", " ".join(args))
    return subprocess.run(args, cwd=cwd, check=check, text=True, capture_output=False)


def git_output(*args, cwd=DATA_ROOT):
    return subprocess.check_output(["git", *args], cwd=cwd, text=True).strip()


def ensure_datalad_clone(url, path, revision, expected_commit):
    """Clone directly, set repository-local identity, repair annex, and pin."""
    path = Path(path)
    if path.exists() and not (path / ".git").exists():
        raise RuntimeError(f"{path} exists but is not a Git dataset; move it aside")
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        run_command(["datalad", "clone", url, path])
    run_command(["git", "config", "--local", "user.name", GIT_IDENTITY_NAME], cwd=path)
    run_command(["git", "config", "--local", "user.email", GIT_IDENTITY_EMAIL], cwd=path)
    annex_check = subprocess.run(["git", "annex", "info"], cwd=path, text=True, capture_output=True)
    if annex_check.returncode != 0:
        run_command(["git", "annex", "init"], cwd=path)
    run_command(["git", "checkout", "--detach", revision], cwd=path)
    actual = git_output("rev-parse", "HEAD", cwd=path)
    assert actual == expected_commit, (path, actual, expected_commit)
    return actual


## 3. Clone OpenNeuro and direct auxiliary datasets

Each clone is lightweight: annexed image content is not downloaded yet. Direct repository URLs avoid the obsolete RIA/subdataset route in the historical StudyForrest superdataset.

In [ ]:
actual_commit = ensure_datalad_clone(
    OPENNEURO_URL, DATA_ROOT, OPENNEURO_TAG, OPENNEURO_COMMIT
)
print(f"Pinned OpenNeuro {OPENNEURO_TAG}:", actual_commit)

Clone the four auxiliary repositories directly and pin them explicitly:

| Dataset | Pinned commit |
|---|---|
| OpenNeuro ds000113 v1.3.0 | `35252aa8e7eab533e7a346d7eb6809aa6081a61d` |
| curated annotations | `155097f944f9dc5a8790dd9148016508ab8b18ad` |
| body contact | `ad42e8aab4a03d0f377ad4db7a9318094f7fc5d4` |
| measured retinotopy maps | `6629c2f2f48e63899e36126087b8deed4ce90153` |
| subject-specific BOLD templates | `0afa47d792c28d493a79ca8854c4ffa1a015347f` |

In [ ]:
auxiliary_pins = [
    (ANNOTATIONS_URL, ANNOTATIONS_ROOT, ANNOTATIONS_COMMIT),
    (BODYCONTACT_URL, BODYCONTACT_ROOT, BODYCONTACT_COMMIT),
    (RETINOTOPY_URL, RETINOTOPY_ROOT, RETINOTOPY_COMMIT),
    (TRANSFORMS_URL, TRANSFORMS_ROOT, TRANSFORMS_COMMIT),
]
for url, path, commit in auxiliary_pins:
    actual = ensure_datalad_clone(url, path, commit, commit)
    print(path.name, actual)

run_command(["datalad", "status", "--annex"], cwd=DATA_ROOT)

## 4. Retrieve one subject's required content

This cell downloads the eight movie runs, annotation tables, functional retinotopy maps, and the `bold3Tp2` registration reference. It does not download all StudyForrest data.

In [ ]:
movie_rel = Path(SUBJECT) / "ses-movie/func"
bold_rel = [
    movie_rel / f"{SUBJECT}_ses-movie_task-movie_run-{run}_bold.nii.gz"
    for run in RUNS
]

annotation_rel = [
    Path("segments/avmovie") / f"emotions_av_1s_events_run-{run}_events.tsv"
    for run in RUNS
] + [
    Path("segments/avmovie") / f"conf_visual_run-{run}_events.tsv"
    for run in RUNS
] + [
    Path("segments/avmovie") / f"conf_audio_run-{run}_events.tsv"
    for run in RUNS
]

bodycontact_rel = [Path("data") / f"obs{i}.csv" for i in range(1, 8)]

run_command(["datalad", "get", *bold_rel], cwd=DATA_ROOT)
run_command(["datalad", "get", *annotation_rel], cwd=ANNOTATIONS_ROOT)
run_command(["datalad", "get", *bodycontact_rel], cwd=BODYCONTACT_ROOT)

retinotopy_rel = [
    Path(SUBJECT) / "post_processing/combined_pipeline_polar.nii.gz",
    Path(SUBJECT) / "post_processing/combined_pipeline_eccentricity.nii.gz",
    Path("qa/pyretmap_subjQuali.ods"),
]
template_rel = Path(SUBJECT) / "bold3Tp2/brain.nii.gz"
run_command(["datalad", "get", *retinotopy_rel], cwd=RETINOTOPY_ROOT)
run_command(["datalad", "get", template_rel], cwd=TRANSFORMS_ROOT)

polar_path = RETINOTOPY_ROOT / retinotopy_rel[0]
eccentricity_path = RETINOTOPY_ROOT / retinotopy_rel[1]
bold3tp2_template = TRANSFORMS_ROOT / template_rel

raw_bold_paths = [DATA_ROOT / path for path in bold_rel]
missing = [str(path) for path in raw_bold_paths if not path.exists()]
assert not missing, f"Missing after datalad get: {missing}"
print(f"Retrieved {len(raw_bold_paths)} OpenNeuro movie runs")

## 5. Motion-align runs and build V1 in subject-native BOLD space

MCFLIRT aligns every OpenNeuro movie run to the first volume of run 1 and emits six motion parameters. FLIRT then estimates a six-degree-of-freedom registration from the released retinotopy `bold3Tp2` template to this movie-BOLD grid.

The retinotopy repository supplies measured polar-angle and eccentricity maps, but no finished V1 boundary. Therefore, draw a bilateral V1 mask manually in the `bold3Tp2` grid by following polar-angle reversals and coherent eccentricity progression. The notebook deliberately stops if that mask is absent; it never substitutes an anatomical atlas label.

The **only source of V1 membership** is the functional polar-angle/eccentricity data in the retinotopy repository. The template-transforms repository is used only to obtain the subject's `bold3Tp2/brain.nii.gz` coordinate reference needed to register that functional mask to the movie; it contributes no ROI, atlas, or anatomical label. Open that reference plus the two functional maps in FSLeyes, create a new binary mask, and save it at the exact path printed by the next cell. This remains subject-specific and uses no MNI, group, or FreeSurfer label.

In [ ]:
preproc_dir = ANALYSIS_ROOT / "preprocessed" / SUBJECT
roi_dir = ANALYSIS_ROOT / "rois" / SUBJECT
preproc_dir.mkdir(parents=True, exist_ok=True)
roi_dir.mkdir(parents=True, exist_ok=True)

bold_reference_3d = preproc_dir / "run-1_volume-0_reference.nii.gz"
if not bold_reference_3d.exists():
    run_command(["fslroi", raw_bold_paths[0], bold_reference_3d, "0", "1"])

bold_paths = []
motion_paths = []
for run, source in zip(RUNS, raw_bold_paths):
    output_stem = preproc_dir / f"{SUBJECT}_task-movie_run-{run}_space-native_desc-moco_bold"
    output_image = output_stem.with_suffix(".nii.gz")
    output_motion = Path(f"{output_stem}.par")
    if not output_image.exists() or not output_motion.exists():
        run_command([
            "mcflirt", "-in", source, "-out", output_stem,
            "-reffile", bold_reference_3d, "-plots", "-mats", "-spline_final",
        ])
    bold_paths.append(output_image)
    motion_paths.append(output_motion)

mean_bold = preproc_dir / f"{SUBJECT}_space-native_desc-mean_bold.nii.gz"
if not mean_bold.exists():
    run_command(["fslmaths", bold_paths[0], "-Tmean", mean_bold])

template_to_movie_mat = roi_dir / f"{SUBJECT}_from-bold3Tp2_to-movieNative_6dof.mat"
template_in_movie = roi_dir / f"{SUBJECT}_space-movieNative_desc-bold3Tp2Template.nii.gz"
if not template_to_movie_mat.exists() or not template_in_movie.exists():
    run_command([
        "flirt", "-in", bold3tp2_template, "-ref", mean_bold,
        "-omat", template_to_movie_mat, "-out", template_in_movie,
        "-dof", "6", "-cost", "normmi",
    ])

# The retinotopy README defines the volumetric maps as being on this bold3Tp2
# voxel lattice. Some released map files store a different NIfTI affine, so
# use the documented reference header after verifying the array dimensions.
source_template = nib.load(bold3tp2_template)
map_components = {}
for name, path in {"polar": polar_path, "eccentricity": eccentricity_path}.items():
    image = nib.load(path)
    values = np.asarray(image.dataobj)
    assert values.ndim == 4 and values.shape[3] >= 2, (name, values.shape)
    assert values.shape[:3] == source_template.shape, (
        f"{name} map and bold3Tp2 reference have different voxel lattices: "
        f"{values.shape[:3]} versus {source_template.shape}"
    )
    if not np.allclose(image.affine, source_template.affine, atol=1e-3):
        print(f"{name}: replacing the released map affine with the documented bold3Tp2 reference affine")
    for component in range(values.shape[3]):
        finite = values[..., component][np.isfinite(values[..., component])]
        print(name, f"component {component} range:", (float(finite.min()), float(finite.max())))
    phase_output = roi_dir / f"{SUBJECT}_space-bold3Tp2_{name}-phase.nii.gz"
    component1_output = roi_dir / f"{SUBJECT}_space-bold3Tp2_{name}-component1.nii.gz"
    reference_header = source_template.header.copy()
    reference_header.set_data_dtype(np.float32)
    nib.save(nib.Nifti1Image(values[..., 0].astype(np.float32), source_template.affine, reference_header), phase_output)
    nib.save(nib.Nifti1Image(values[..., 1].astype(np.float32), source_template.affine, reference_header), component1_output)
    map_components[name] = {"phase": phase_output, "component1": component1_output}

manual_v1_source = roi_dir / f"{SUBJECT}_space-bold3Tp2_desc-retinotopyV1_mask.nii.gz"
print("\nDraw bilateral V1 from the functional maps and save it as:")
print(manual_v1_source)
print("\nFSLeyes inputs:")
print(bold3tp2_template)
print(map_components["polar"]["phase"])
print(map_components["eccentricity"]["phase"])
print("Optional response/statistic components for thresholding:")
print(map_components["polar"]["component1"])
print(map_components["eccentricity"]["component1"])

if not manual_v1_source.exists():
    raise FileNotFoundError(
        "Manual functional V1 mask not found. Use FSLeyes to delineate V1 from "
        "the polar-angle reversals and eccentricity progression, save it at the "
        f"printed path, then rerun this cell: {manual_v1_source}"
    )

source_mask = nib.load(manual_v1_source)
assert source_mask.shape == source_template.shape, "Manual mask/template shape mismatch"
assert np.allclose(source_mask.affine, source_template.affine, atol=1e-3), "Manual mask/template affine mismatch"

roi_path = roi_dir / f"{SUBJECT}_space-movieNative_desc-retinotopyV1_mask.nii.gz"
run_command([
    "flirt", "-in", manual_v1_source, "-ref", mean_bold,
    "-applyxfm", "-init", template_to_movie_mat,
    "-interp", "nearestneighbour", "-out", roi_path,
])
run_command(["fslmaths", roi_path, "-bin", roi_path])
roi_label = "bilateral V1 manually delineated from measured retinotopy"

print(roi_label)
print(roi_path)

In [ ]:
ref_img = nib.load(bold_reference_3d)
roi_img = nib.load(roi_path)
assert roi_img.shape == ref_img.shape[:3], (roi_img.shape, ref_img.shape)
assert np.allclose(roi_img.affine, ref_img.affine, atol=1e-3), "ROI/BOLD affine mismatch"
n_voxels = int(np.count_nonzero(np.asarray(roi_img.dataobj)))
assert n_voxels > 0
for path in bold_paths:
    image = nib.load(path)
    assert image.shape[:3] == ref_img.shape, (path, image.shape, ref_img.shape)
    assert np.allclose(image.affine, ref_img.affine, atol=1e-3), f"Grid mismatch: {path}"
print("ROI voxels:", n_voxels)

registration_display = plot_stat_map(
    template_in_movie, bg_img=mean_bold, threshold=None,
    title=f"{SUBJECT}: bold3Tp2 template registered to movie-native BOLD",
)
registration_display.show()

plot_display = plot_roi(roi_img, bg_img=ref_img, title=f"{SUBJECT}: {roi_label}")
plot_display.show()

## 6. Convert annotations to one feature row per TR

The released emotion and low-level tables already use run-relative audio-visual timing. Body-contact annotations use continuous research-cut timing, so they are split using the official movie-segment starts.

Body contact is summarized as the mean fraction of each TR covered by contact across seven observers. This preserves agreement information and prevents duplicate actor/recipient rows from inflating a single observer's contribution.

In [ ]:
SEGMENT_STARTS = np.array([0.00, 886.00, 1752.08, 2612.16, 3572.20, 4480.28, 5342.36, 6410.44])
LOWLEVEL_VISUAL = ["brmean", "brlr", "brud", "norm_diff"]
LOWLEVEL_AUDIO = ["rms", "lrdiff"]
EMOTION_COLUMNS = [
    "arousal", "valence_positive", "valence_negative",
    "e_admiration", "e_anger/rage", "e_contempt", "e_disappointment",
    "e_fear", "e_gratitude", "e_happiness", "e_hate", "e_hope",
    "e_love", "e_pity/compassion", "e_pride", "e_relief",
    "e_remorse", "e_resent", "e_sadness", "e_satisfaction", "e_shame",
]


def overlap_features(events, n_tr, columns, tr=TR, add_presence=False):
    """Duration-weighted feature means in TR bins; supports overlapping events."""
    values = np.zeros((n_tr, len(columns)), dtype=np.float64)
    weights = np.zeros(n_tr, dtype=np.float64)
    numeric = events[["onset", "duration", *columns]].apply(pd.to_numeric, errors="coerce")
    for row in numeric.itertuples(index=False, name=None):
        start = float(row[0])
        end = start + float(row[1])
        if end <= 0 or start >= n_tr * tr:
            continue
        vector = np.asarray(row[2:], dtype=float)
        vector = np.nan_to_num(vector)
        first = max(0, int(np.floor(start / tr)))
        last = min(n_tr - 1, int(np.ceil(end / tr) - 1))
        for index in range(first, last + 1):
            overlap = max(0.0, min(end, (index + 1) * tr) - max(start, index * tr))
            values[index] += overlap * vector
            weights[index] += overlap
    valid = weights > 0
    values[valid] /= weights[valid, None]
    if add_presence:
        values = np.column_stack([values, np.clip(weights / tr, 0, 1)])
    return values


def body_contact_by_run(observer_tables, run_index, n_tr, tr=TR):
    """Observer-agreement contact coverage for one run (run_index is zero-based)."""
    run_start = SEGMENT_STARTS[run_index]
    run_end = run_start + n_tr * tr
    per_observer = np.zeros((len(observer_tables), n_tr), dtype=float)
    for observer_index, table in enumerate(observer_tables):
        for row in table.itertuples(index=False):
            original_start = float(row.start)
            # Equal timestamps denote a sub-second event in this annotation.
            original_end = max(float(row.end), original_start + 1.0)
            start = max(original_start, run_start) - run_start
            end = min(original_end, run_end) - run_start
            if end <= 0 or start >= n_tr * tr or end <= start:
                continue
            first = max(0, int(np.floor(start / tr)))
            last = min(n_tr - 1, int(np.ceil(end / tr) - 1))
            for index in range(first, last + 1):
                overlap = max(0.0, min(end, (index + 1) * tr) - max(start, index * tr))
                per_observer[observer_index, index] += overlap / tr
        per_observer[observer_index] = np.clip(per_observer[observer_index], 0, 1)
    return per_observer.mean(axis=0, keepdims=True).T

In [ ]:
observer_tables = [
    pd.read_csv(BODYCONTACT_ROOT / "data" / f"obs{i}.csv")
    for i in range(1, 8)
]

run_features = []
feature_names = None

for run, bold_path in zip(RUNS, bold_paths):
    n_tr = nib.load(bold_path).shape[3]
    segment_dir = ANNOTATIONS_ROOT / "segments/avmovie"

    visual = pd.read_csv(segment_dir / f"conf_visual_run-{run}_events.tsv", sep="\t")
    audio = pd.read_csv(segment_dir / f"conf_audio_run-{run}_events.tsv", sep="\t")
    emotion = pd.read_csv(segment_dir / f"emotions_av_1s_events_run-{run}_events.tsv", sep="\t")

    # Some emotion columns may be absent in future revisions; retain available columns explicitly.
    emotion_cols = [column for column in EMOTION_COLUMNS if column in emotion.columns]
    base = np.column_stack([
        overlap_features(visual, n_tr, LOWLEVEL_VISUAL),
        overlap_features(audio, n_tr, LOWLEVEL_AUDIO),
    ])
    emotion_x = overlap_features(emotion, n_tr, emotion_cols, add_presence=True)
    contact_x = body_contact_by_run(observer_tables, run - 1, n_tr)

    run_features.append({"base": base, "emotion": emotion_x, "contact": contact_x})
    if feature_names is None:
        feature_names = {
            "base": LOWLEVEL_VISUAL + LOWLEVEL_AUDIO,
            "emotion": emotion_cols + ["emotion_presence"],
            "contact": ["body_contact_agreement"],
        }
    print(f"run {run}: {n_tr} TRs; base={base.shape}, emotion={emotion_x.shape}, contact={contact_x.shape}")

assert all(np.isfinite(block).all() for run in run_features for block in run.values())

In [ ]:
fig, axes = plt.subplots(len(RUNS), 1, figsize=(14, 10), sharex=False, constrained_layout=True)
for run, features, axis in zip(RUNS, run_features, axes):
    axis.plot(features["emotion"][:, -1], label="emotion presence", lw=1)
    axis.plot(features["contact"][:, 0], label="body-contact agreement", lw=1)
    axis.set_ylabel(f"run {run}")
axes[0].legend(loc="upper right")
axes[-1].set_xlabel("TR")
plt.show()

## 7. Extract native-space V1 responses

Each run is detrended, high-pass filtered, and standardized independently. Motion estimates are supplied as nuisance regressors. Confirm the number and convention of motion columns before publication; the released files are raw MCFLIRT-style estimates.

In [ ]:
masker = NiftiMasker(
    mask_img=roi_path,
    detrend=True,
    standardize="zscore_sample",
    high_pass=0.008,
    t_r=TR,
    smoothing_fwhm=None,
)

run_responses = []
for run, bold_path, motion_path in zip(RUNS, bold_paths, motion_paths):
    motion = np.loadtxt(motion_path)
    response = masker.fit_transform(bold_path, confounds=motion)
    assert response.shape[0] == run_features[run - 1]["base"].shape[0]
    run_responses.append(response)
    print(f"run {run}: response {response.shape}, motion {motion.shape}")

print("Total finite response values:", sum(np.isfinite(y).sum() for y in run_responses))

## 8. FIR-lagged nested models

A small finite impulse response basis (4, 6, 8 s) avoids assuming a single exact HRF. Lags are created **within each run** so information never crosses run boundaries.

Models compared:

- `base`: low-level brightness/spatial-difference/change plus audio RMS/lateral difference;
- `base+emotion`;
- `base+contact`;
- `full`: base + emotion + contact.

The score is Pearson correlation between predicted and observed held-out time series for each voxel. The key quantities are paired incremental scores such as `full − base`, not training fit or raw coefficient size.

In [ ]:
MODEL_BLOCKS = {
    "base": ("base",),
    "base+emotion": ("base", "emotion"),
    "base+contact": ("base", "contact"),
    "full": ("base", "emotion", "contact"),
}


def lag_within_run(matrix, lags=HRF_LAGS_TR):
    max_lag = max(lags)
    lagged = np.hstack([matrix[max_lag - lag : len(matrix) - lag] for lag in lags])
    return lagged, max_lag


designs = {name: [] for name in MODEL_BLOCKS}
responses = []
for features, response in zip(run_features, run_responses):
    for model_name, blocks in MODEL_BLOCKS.items():
        raw_design = np.column_stack([features[block] for block in blocks])
        lagged, dropped = lag_within_run(raw_design)
        designs[model_name].append(lagged)
    responses.append(response[dropped:])

for name in MODEL_BLOCKS:
    print(name, [x.shape for x in designs[name]])

In [ ]:
def columnwise_correlation(observed, predicted):
    observed = observed - observed.mean(axis=0, keepdims=True)
    predicted = predicted - predicted.mean(axis=0, keepdims=True)
    numerator = np.sum(observed * predicted, axis=0)
    denominator = np.sqrt(np.sum(observed**2, axis=0) * np.sum(predicted**2, axis=0))
    return np.divide(numerator, denominator, out=np.full_like(numerator, np.nan), where=denominator > 0)


def leave_one_run_out_scores(run_designs, run_targets, alpha=RIDGE_ALPHA):
    fold_scores = []
    fold_predictions = []
    for test_index in range(len(run_designs)):
        train_indices = [i for i in range(len(run_designs)) if i != test_index]
        x_train = np.vstack([run_designs[i] for i in train_indices])
        y_train = np.vstack([run_targets[i] for i in train_indices])
        x_test = run_designs[test_index]
        y_test = run_targets[test_index]

        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        model.fit(x_train, y_train)
        prediction = model.predict(x_test)
        fold_predictions.append(prediction)
        fold_scores.append(columnwise_correlation(y_test, prediction))
    return np.stack(fold_scores), fold_predictions


scores = {}
predictions = {}
for model_name, run_designs in designs.items():
    scores[model_name], predictions[model_name] = leave_one_run_out_scores(run_designs, responses)
    print(model_name, "mean held-out r =", np.nanmean(scores[model_name]))

`RIDGE_ALPHA` is fixed here to keep the example tractable. For the final analysis, select it using inner run-wise cross-validation inside every outer training fold. Never tune it on the held-out run.

In [ ]:
mean_scores = {name: np.nanmean(value, axis=0) for name, value in scores.items()}
delta_emotion = mean_scores["base+emotion"] - mean_scores["base"]
delta_contact = mean_scores["base+contact"] - mean_scores["base"]
delta_joint = mean_scores["full"] - mean_scores["base"]

summary = pd.DataFrame({
    "base_r": mean_scores["base"],
    "emotion_delta_r": delta_emotion,
    "contact_delta_r": delta_contact,
    "joint_delta_r": delta_joint,
})
display(summary.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, column, title in zip(
    axes,
    ["emotion_delta_r", "contact_delta_r", "joint_delta_r"],
    ["Emotion − base", "Contact − base", "Full − base"],
):
    sns.histplot(summary[column], bins=50, ax=axis)
    axis.axvline(0, color="black", lw=1)
    axis.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
delta_maps = {
    "emotion": masker.inverse_transform(delta_emotion),
    "contact": masker.inverse_transform(delta_contact),
    "joint": masker.inverse_transform(delta_joint),
}

for name, image in delta_maps.items():
    output = ANALYSIS_ROOT / f"{SUBJECT}_{name}_incremental_prediction_in_native_bold.nii.gz"
    image.to_filename(output)
    plot_display = plot_stat_map(
        image,
        bg_img=ref_img,
        title=f"{SUBJECT}: Δ held-out r, {name}",
        symmetric_cbar=True,
    )
    plot_display.show()
    print("Saved", output)

## 9. Inference: preserve temporal autocorrelation

Do not interpret voxels with positive `Δr` as significant from this plot alone. Emotion and contact occur in long, autocorrelated episodes and correlate with visual content.

A defensible test should:

1. circularly shift the **auxiliary feature block independently within each run**, with a minimum shift larger than the HRF window;
2. refit the entire outer-CV comparison for each permutation;
3. use a voxelwise or maximum-statistic null distribution;
4. control multiplicity across V1 voxels and across the emotion/contact hypotheses;
5. replicate effects across subjects, reporting subject-level estimates rather than pooling native-space voxels.

Also compare against a stronger visual baseline. If adding CNN or motion-energy features removes an apparent emotion/contact gain, the original gain likely reflected unmodeled visual structure rather than high-level modulation.

In [ ]:
def circularly_shift_auxiliary(run_feature_blocks, rng, minimum_shift_tr=15):
    """Return copied features with emotion/contact shifted inside each run."""
    shifted = []
    for blocks in run_feature_blocks:
        copied = {name: value.copy() for name, value in blocks.items()}
        n_tr = len(copied["base"])
        allowed = np.arange(minimum_shift_tr, n_tr - minimum_shift_tr)
        if len(allowed) == 0:
            raise ValueError("Run is too short for the requested minimum shift")
        for name in ("emotion", "contact"):
            copied[name] = np.roll(copied[name], int(rng.choice(allowed)), axis=0)
        shifted.append(copied)
    return shifted


# Sanity-check the permutation generator. Full refitting is intentionally not
# launched automatically because hundreds/thousands of fits can be expensive.
rng = np.random.default_rng(2026)
permuted_features = circularly_shift_auxiliary(run_features, rng)
assert all(np.array_equal(a["base"], b["base"]) for a, b in zip(run_features, permuted_features))
assert any(not np.array_equal(a["contact"], b["contact"]) for a, b in zip(run_features, permuted_features))
print("Permutation generator passed its checks.")

## 10. Recommended next improvements

- Extract frame-wise motion-energy, Gabor-pyramid, or pretrained-video-model features and add them to `base`.
- Add eye position and saccade regressors: retinal stimulation is especially important for V1.
- Tune ridge regularization with nested run-wise CV.
- Compare multiple HRF/FIR choices without using the final test runs for selection.
- Recheck the manual V1 boundary against both polar angle and eccentricity and document the delineation rule.
- Save derived designs, masks, and result maps in a new DataLad analysis dataset with `datalad run` provenance.

## Data provenance and citations

- OpenNeuro ds000113 Git/DataLad dataset: <https://github.com/OpenNeuroDatasets/ds000113>
- Curated annotations: <https://github.com/psychoinformatics-de/studyforrest-data-annotations>
- Body-contact annotations: <https://github.com/psychoinformatics-de/studyforrest-paper-bodycontactannotation>
- Image-space transforms: <https://github.com/psychoinformatics-de/studyforrest-data-templatetransforms>
- Retinotopy maps: <https://github.com/psychoinformatics-de/studyforrest-data-retinotopy>

Cite the relevant StudyForrest acquisition, annotation, and derivative publications in addition to the repositories.